# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
from collections import defaultdict
from tqdm.notebook import trange, tqdm
from icecream import ic

import pandas as pd
import numpy as np
import scipy
import sklearn
import prince

import seaborn as sns
from matplotlib import pyplot as plt

from plotly import express as px
from plotly import graph_objects as go
import plotly.io as pio

In [ ]:
from rsfp.metrics import *
from rsfp.constants import *
from rsfp.data import *
from rsfp.matching import *
from rsfp.dimensionality_reduction import *
from rsfp.utils import *

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
pd.options.plotting.backend = "plotly"

# Data

## 2023

In [ ]:
# load 2023 datasets
df_voters, df_candidates, df_questions = build_all(verbose=True)

In [ ]:
# load uncleaned 2023 voter dataset
df_voters_all = build_voters(clean=False)
df_voters_all

In [ ]:
# load recommendation data for L2 distance metric
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load recommendation data for evaluated distance metrics
df_voters = df_voters.load_candidate_voting_recommendations('df_voters_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters = df_voters.load_list_voting_recommendations('df_voters_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# add dimensionality reduction columns
df_candidates, obj = add_pca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_pca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_ca_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_ca_cols(df_voters, n_dims=2, obj=obj)

df_candidates, obj = add_tsvd_cols(df_candidates, n_dims=2, return_obj=True)
df_voters = add_tsvd_cols(df_voters, n_dims=2, obj=obj)

In [ ]:
# plot PCA distribution of candidates
fig = px.scatter(df_candidates[~df_candidates['_party'].isin(['Parteilos', 'Andere'])], x='PCA_0', y='-PCA_1', color='_party', color_discrete_map=PARTY2COLOR)
fig.update_layout(
    xaxis_title='Left - Right',
    yaxis_title='Conservative - Liberal',
    legend=dict(
        title='Party',
        x=1,
        y=1,
        xanchor='left',
        yanchor='top',
        itemsizing='constant',
    )
).update_traces(marker_size=3)


## 2019

In [ ]:
# load 2019 datasets
df_voters19, df_candidates19, df_questions19 = build_all19(verbose=True)

In [ ]:
# load uncleaned 2019 voter dataset
df_voters19_all = build_voters19(clean=False)

In [ ]:
# load recommendation data for L2 distance metric
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load list recommendations for L2 distance metric
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=['L2_sv'])

In [ ]:
# load recommendation data for evaluated distance metrics
df_voters19 = df_voters19.load_candidate_voting_recommendations('df_voters19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)
df_candidates19 = df_candidates19.load_candidate_recommendation_counts('df_candidates19_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# load list recommendations for evaluated distance metrics
df_voters19 = df_voters19.load_list_voting_recommendations('df_voters19_list_recommendations_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# add dimensionality reduction columns
df_candidates19, obj = add_pca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_pca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_ca_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_ca_cols(df_voters19, n_dims=2, obj=obj)

df_candidates19, obj = add_tsvd_cols(df_candidates19, n_dims=2, return_obj=True)
df_voters19 = add_tsvd_cols(df_voters19, n_dims=2, obj=obj)

# Answer Calibration (AC + CP)

## Neutral Answers (CP-M)

In [ ]:
# HELPER FUNCTIONS

def map_to_closest_old(array_2d, specified_numbers):
    # Expand dimensions to perform broadcasting
    array_expanded = np.expand_dims(array_2d, axis=-1)  # Shape (m, n, 1)
    specified_expanded = np.expand_dims(specified_numbers, axis=(0, 1))  # Shape (1, 1, k)
    
    # Calculate the absolute differences between array elements and specified numbers
    differences = np.abs(array_expanded - specified_expanded)  # Shape (m, n, k)
    
    # Find the index of the minimum difference
    closest_indices = np.argmin(differences, axis=-1)  # Shape (m, n)
    
    # Map to the closest number
    mapped_array = specified_numbers[closest_indices]  # Shape (m, n)
    
    return mapped_array


def map_to_closest(array_2d, specified_numbers):
    # Create an array to store the result
    mapped_array = np.full(array_2d.shape, np.nan)

    # Find non-NaN elements
    non_nan_mask = ~np.isnan(array_2d)
    
    # Expand dimensions to perform broadcasting for non-NaN elements
    array_expanded = np.expand_dims(array_2d[non_nan_mask], axis=-1)  # Shape (p, 1)
    specified_expanded = np.expand_dims(specified_numbers, axis=0)  # Shape (1, k)
    
    # Calculate the absolute differences between array elements and specified numbers
    differences = np.abs(array_expanded - specified_expanded)  # Shape (p, k)
    
    # Find the index of the minimum difference
    closest_indices = np.argmin(differences, axis=-1)  # Shape (p,)
    
    # Map to the closest number
    mapped_array[non_nan_mask] = specified_numbers[closest_indices]  # Shape (m, n)
    
    return mapped_array


def map_to_answer_possibilites(answer_matrix):
    assert answer_matrix.shape[1] == 75, "answer matrix has to have 75 columns corresponding to the 75 questions in 2023 questionnaire"
    
    answer_matrix[:, :60] = map_to_closest(answer_matrix[:, :60], np.array([0, 25, 75, 100]))
    answer_matrix[:, 60:67] = map_to_closest(answer_matrix[:, 60:67], np.array([0, 17, 33, 50, 67, 83, 100]))
    answer_matrix[:, 67:] = map_to_closest(answer_matrix[:, 67:], np.array([0, 25, 50, 75, 100]))

    return answer_matrix
    


def get_simulated_gaussian_sample(df_candidates_or_voters: pd.DataFrame, size=1000, map_to_answer_possibilities: bool = True, seed: int = 0):
    empirical_mean = df_candidates_or_voters.a().mean().to_numpy()
    empirical_cov = np.cov(df_candidates_or_voters.a(), rowvar=False)

    rng = np.random.default_rng(seed=seed)
    sample = rng.multivariate_normal(mean=empirical_mean, cov=empirical_cov, size=size)

    if map_to_answer_possibilities:
        sample = map_to_answer_possibilites(sample)

    return sample

In [ ]:
# COMPUTE WHAT-IF ANALYSIS (ALL CANDIDATES OF A PARTY EMPLOY MODERATE ANSWERING STRATEGY)

method_dfs = {
    
}

for method in tqdm([m for m in EVAL_DISTANCE_METHODS if m!='L2_sv']):
    ic(method)
    method_dfs[method] = {
        'candidates': {},
    }
    for party in tqdm(df_candidates['_party'].unique()):
        ic(party)
        df_candidates_modified = df_candidates.copy()
        df_voters_recommendations = df_voters.copy()
        candidate_party_mask = df_candidates_modified['_party'] == party
    
        # minimize answer strengths of all candidates in party
        df_candidates_modified.loc[candidate_party_mask, get_cols(df_candidates_modified)] = map_to_answer_possibilites(((df_candidates_modified[candidate_party_mask].a().to_numpy() - 50)*0.01)+50)
    
        # update answer strength stats
        df_candidates_modified.loc[candidate_party_mask, '_answer_strength'] = (df_candidates_modified[candidate_party_mask][get_cols(df_candidates_modified)] - 50).abs().mean(axis=1)
        df_candidates_modified.loc[candidate_party_mask, '_answer_strength_std'] = (df_candidates_modified[candidate_party_mask][get_cols(df_candidates_modified)] - 50).abs().std(axis=1)
    
        # calculate what if recommendations
        df_voters_recommendations = add_candidate_voting_recommendations(df_voters_recommendations, df_candidates_modified, distance_method=method)
    
        df_candidates_modified.add_recommendation_counts(df_voters_recommendations)
    
        ic(df_candidates_modified.columns[-5:])
    
        # store results
        method_dfs[method]['candidates'][party] = df_candidates_modified

    filename = 'data/cache/what_if_analysis_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
    with open(filename, 'wb') as file:
        # Serialize the dictionary and write it to the file
        pickle.dump(method_dfs, file)

In [ ]:
# LOAD
import pickle

filename = 'data/cache/what_if_analysis_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
with open(filename, 'rb') as file:
    # Serialize the dictionary and write it to the file
    method_dfs = pickle.load(file)

In [ ]:
# SAVE

filename = 'data/cache/what_if_analysis_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
with open(filename, 'wb') as file:
    # Serialize the dictionary and write it to the file
    pickle.dump(method_dfs, file)

In [ ]:
# load smartvote recommendation counts
df_candidates = df_candidates.load_candidate_recommendation_counts('df_candidates_recommendations_spc_10_methods', selected_distance_methods=EVAL_DISTANCE_METHODS)

In [ ]:
# ANALYZE

#party_col = 'party_short'
#unique_parties = df_candidates[party_col].unique()

#party_col, unique_parties = '_party', BIG_PARTIES

party_col, unique_parties = '_party', df_candidates['_party'].unique()

dfs_analysis = []
for method in method_dfs:
    manipulated_strengths = []
    for party in unique_parties:
        print(party)
        manipulated_strengths.append(method_dfs[method]['candidates'][party].get_party_visibilities(party_col=party_col).loc[party, f'Party Visibility ({method})'])

    df_party_strengths = df_candidates.get_party_visibilities(party_col=party_col)
    df_analysis = df_party_strengths[f'Party Visibility ({method})'].to_frame().join(pd.DataFrame(index=unique_parties, data=manipulated_strengths, columns=[f'Manipulated ({method})'])).rename(columns={f'Party Visibility ({method})': f'Actual ({method})'})
    dfs_analysis.append(df_analysis)

df_analysis = pd.concat(dfs_analysis, axis=1)
df_analysis

In [ ]:
df_candidates = df_candidates.add_n_votes()

df_candidates.get_party_visibilities()

In [ ]:
for method in np.unique([c.split('(')[1].split(')')[0] for c in df_analysis.columns if '(' in c]):
    df_analysis[f'Ratio ({method})'] = ((df_analysis[f'Manipulated ({method})'] / df_analysis[f'Actual ({method})']))
    df_analysis[f'Gain ({method})'] = ((df_analysis[f'Manipulated ({method})'] - df_analysis[f'Actual ({method})']))

In [ ]:
df_analysis['Party Visibility'] = df_analysis.index.map(df_candidates.get_party_visibilities()['Party Visibility (Actual Votes)'].to_dict())

df_analysis_parties = df_analysis.drop(labels=['Andere', 'Parteilos'])
df_analysis_parties

In [ ]:
(df_analysis_parties[[c for c in df_analysis.columns if c.startswith('Ratio')]] * df_analysis_parties['Party Visibility'].to_numpy().reshape(-1, 1)).sum() / df_analysis_parties['Party Visibility'].sum()

### Metric (CP-M)

In [ ]:
# CP-M

(df_analysis_parties[[c for c in df_analysis_parties.columns if c.startswith('Ratio')]] * df_analysis_parties['Party Visibility'].to_numpy().reshape(-1, 1)).sum() / df_analysis_parties['Party Visibility'].sum() - 1

### Plot

In [ ]:
# VISUALIZE
method = 'L1'
methods = ['L1', 'L2_sv']

#custom_colors = ['#b3b3b3', '#ffcc80', '#ff9900']
custom_colors = ['#cbcbcb','#ffb442', '#858585', '#f06542']

fig = px.bar(100*df_analysis.loc[BIG_PARTIES, [i for method in methods for i in [f'Actual ({method})', f'Manipulated ({method})']]], barmode='group', color_discrete_sequence=custom_colors)

trace_name_map = {
    'Actual (L2_sv)': 'Actual (L2)', 
    'Manipulated (L2_sv)': 'Calibrated (L2)',
    'Manipulated (L1)': 'Calibrated (L1)',
    'Manipulated (angular)': 'Calibrated (Angular)',
}

fig.for_each_trace(lambda trace: trace.update(name=trace_name_map.get(trace.name) if trace.name in trace_name_map else trace.name))

fig.update_layout(
    yaxis_title='Visibility (%)',
    xaxis_title=None,
    legend=dict(
        title=None,
        x=1,
        y=1,
        xanchor='right',
        yanchor='top',
    )
)



In [ ]:
save_paper_figure(fig, 'what_if_neutral_answering_strategy_recommendation_share_L2_L1_big_parties_2023_sv_tiebreaking_times_roman_5x3_both_actual_times_bottommargin_english', height=360, font_size=18, margin_b=50)

## Extreme Answers (CP-E)

In [ ]:
# COMPUTE WHAT-IF ANALYSIS (ALL CANDIDATES OF A PARTY EMPLOY EXTREME ANSWERING STRATEGY)

#method_dfs = {}

for method in tqdm([m for m in EVAL_DISTANCE_METHODS[1:]]):
    ic(method)
    method_dfs[method] = {
        'candidates': {},
    }
    for party in tqdm(df_candidates['_party'].unique()):
        ic(party)
        df_candidates_modified = df_candidates.copy()
        df_voters_recommendations = df_voters.copy()
        candidate_party_mask = df_candidates_modified['_party'] == party
    
        # maximize answer strengths of all candidates in party
        df_candidates_modified.loc[candidate_party_mask, get_cols(df_candidates_modified)] = map_to_answer_possibilites(((df_candidates_modified[candidate_party_mask].a().to_numpy() - 50)*100)+50)
    
        # update answer strength stats
        df_candidates_modified.loc[candidate_party_mask, '_answer_strength'] = (df_candidates_modified[candidate_party_mask][get_cols(df_candidates_modified)] - 50).abs().mean(axis=1)
        df_candidates_modified.loc[candidate_party_mask, '_answer_strength_std'] = (df_candidates_modified[candidate_party_mask][get_cols(df_candidates_modified)] - 50).abs().std(axis=1)
    
        # calculate what if recommendations
        df_voters_recommendations = add_candidate_voting_recommendations(df_voters_recommendations, df_candidates_modified, distance_method=method)
    
        df_candidates_modified.add_recommendation_counts(df_voters_recommendations)
    
        ic(df_candidates_modified.columns[-5:])
    
        # store results
        method_dfs[method]['candidates'][party] = df_candidates_modified

    filename = 'data/cache/what_if_analysis_extreme_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
    with open(filename, 'wb') as file:
        # Serialize the dictionary and write it to the file
        pickle.dump(method_dfs, file)

In [ ]:
# LOAD

filename = 'data/cache/what_if_analysis_extreme_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
with open(filename, 'rb') as file:
    # Serialize the dictionary and write it to the file
    method_dfs = pickle.load(file)

In [ ]:
# SAVE

import pickle

filename = 'data/cache/what_if_analysis_extreme_all_party_dfs_eval_methods_sv_tiebreaking_recommendations.pkl'
with open(filename, 'wb') as file:
    # Serialize the dictionary and write it to the file
    pickle.dump(method_dfs, file)

In [ ]:
# ANALYZE

#party_col = 'party_short'
#unique_parties = df_candidates[party_col].unique()

#party_col, unique_parties = '_party', BIG_PARTIES

party_col, unique_parties = '_party', df_candidates['_party'].unique()

dfs_analysis = []
for method in method_dfs:
    manipulated_strengths = []
    for party in unique_parties:
        manipulated_strengths.append(method_dfs[method]['candidates'][party].get_party_visibilities(party_col=party_col).loc[party, f'Party Visibility ({method})'])

    df_party_strengths = df_candidates.get_party_visibilities(party_col=party_col)
    df_analysis = df_party_strengths[f'Party Visibility ({method})'].to_frame().join(pd.DataFrame(index=unique_parties, data=manipulated_strengths, columns=[f'Manipulated ({method})'])).rename(columns={f'Party Visibility ({method})': f'Actual ({method})'})
    dfs_analysis.append(df_analysis)

df_analysis = pd.concat(dfs_analysis, axis=1)

for method in np.unique([c.split('(')[1].split(')')[0] for c in df_analysis.columns if '(' in c]):
    df_analysis[f'Ratio ({method})'] = ((df_analysis[f'Manipulated ({method})'] / df_analysis[f'Actual ({method})']))
    df_analysis[f'Gain ({method})'] = ((df_analysis[f'Manipulated ({method})'] - df_analysis[f'Actual ({method})']))

df_analysis

In [ ]:
df_analysis.style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_n_votes()

In [ ]:
df_analysis['Vote Share'] = df_analysis.index.map(df_candidates.get_party_visibilities()['Party Visibility (Actual Votes)'].to_dict())
df_analysis_parties = df_analysis.drop(index=['Andere', 'Parteilos'])
df_analysis_parties

In [ ]:
df_analysis[[c for c in df_analysis.columns if 'Gain' in c]].mean()

In [ ]:
df_analysis.loc[BIG_PARTIES, 'Ratio (L2_sv)'] - 1

In [ ]:
(df_analysis_parties[[c for c in df_analysis.columns if c.startswith('Ratio')]] * df_analysis_parties['Vote Share'].to_numpy().reshape(-1, 1)).sum() / df_analysis_parties['Vote Share'].sum() - 1


### Metric (CP-E)

In [ ]:
# CP-E

((df_analysis_parties[[c for c in df_analysis.columns if c.startswith('Ratio')]] * df_analysis_parties['Vote Share'].to_numpy().reshape(-1, 1)).sum() / df_analysis_parties['Vote Share'].sum() - 1).to_frame().loc[[f'Ratio ({method})' for method in EVAL_DISTANCE_METHODS]]

# Question Favoritism (QF)

In [ ]:
# COMPUTE (only deluxe voters)

df_greedy_forward_selection = pd.DataFrame()

district = 'SG'
df_v = df_voters[(df_voters['_district']==district) & (df_voters['questTYPE']==1)]
df_c = df_candidates.district(district)

for party in tqdm([p for p in df_c['_party'].unique() if p not in [c.split()[0] for c in df_greedy_forward_selection.columns] and p not in ['Andere', 'Parteilos']]):
    ic(party)
    selected_cols = []
    party_strengths = []

    for _ in tqdm(range(75)):
        best_curr_cols_party_strength = 0
        available_cols = [c for c in range(75) if c not in selected_cols]
    
        for col in tqdm(available_cols):
            curr_cols = list(selected_cols + [col])
            curr_cols_bool_index = np.zeros(75)
            curr_cols_bool_index[curr_cols] = 1
            curr_cols_bool_index = curr_cols_bool_index.astype(bool)
    
            df_v_curr_cols = SVDataFrame(df_v, answer_cols=np.array(get_cols(df_v))[curr_cols_bool_index], weight_cols=np.array(get_cols(df_v, col_type='weight'))[curr_cols_bool_index])
            df_c_curr_cols = SVDataFrame(df_c, answer_cols=np.array(get_cols(df_c))[curr_cols_bool_index])

            df_v_curr_cols = add_candidate_voting_recommendations(df_v_curr_cols, df_c_curr_cols, progress_bar=False)
            df_c_curr_cols = df_c_curr_cols.add_recommendation_counts(df_v_curr_cols)
    
            curr_cols_party_strength = df_c_curr_cols.get_party_visibilities().loc[party].item()
            #ic(curr_cols_party_strength)
    
            if curr_cols_party_strength > best_curr_cols_party_strength:
                best_curr_cols_party_strength = curr_cols_party_strength
                next_selected_cols = curr_cols.copy()
        
        selected_cols = next_selected_cols.copy()
        party_strengths.append(best_curr_cols_party_strength)
        ic(best_curr_cols_party_strength)
        #ic(selected_cols)
    
    df_greedy_forward_selection[f"{party} selection"] = selected_cols
    df_greedy_forward_selection[f"{party} strength"] = party_strengths

    # SAVE
    #df_greedy_forward_selection.to_csv(f'df_greedy_forward_selection_{district}_deluxe.csv')

In [ ]:
# Load Greedy Forward Selection Data

#df_greedy_forward_selection = pd.read_csv('df_greedy_forward_selection_SG_all.csv', index_col=0)
df_greedy_forward_selection = pd.read_csv('df_greedy_forward_selection_SG_deluxe.csv', index_col=0)

In [ ]:
for c in [c for c in df_greedy_forward_selection.columns if c.endswith('strength')]:
    df_greedy_forward_selection[f"{c.split()[0]}"] = 100 * (df_greedy_forward_selection[c] / df_greedy_forward_selection[c].min() - 1)
    df_greedy_forward_selection[f"{c.split()[0]} diff"] = df_greedy_forward_selection[c] - df_greedy_forward_selection[c].min()

df_greedy_forward_selection['# Questions'] = df_greedy_forward_selection.index+1

df_greedy_forward_selection[[c.split()[0] for c in df_greedy_forward_selection.columns if c.endswith('strength')]+['# Questions']].melt(id_vars='# Questions', var_name='Party', value_name='Party Visibility')

In [ ]:
df_melted = df_greedy_forward_selection[[c.split()[0] for c in df_greedy_forward_selection.columns if c.endswith('strength')]+['# Questions']].melt(id_vars='# Questions', var_name='Party', value_name='Party Visibility')

df_melted

### Plot

In [ ]:
# VISUALIZE

fig = px.line(
    df_melted,
    x='# Questions',
    y='Party Visibility', 
    color='Party',
    width=1200, 
    height=800,
    color_discrete_map=PARTY2COLOR,
    category_orders={'Party': PARTIES_LEFT_TO_RIGHT},
)



# Find the maximum y-value for each group in 'variable' and add a cross marker
for variable in df_melted['Party'].unique():
    max_row = df_melted[df_melted['Party'] == variable].loc[df_melted[df_melted['Party'] == variable]['Party Visibility'].idxmax()]
    fig.add_trace(go.Scatter(
        x=[max_row['# Questions']],
        y=[max_row['Party Visibility']],
        mode='markers',
        marker=dict(symbol='circle', size=8, color=PARTY2COLOR[variable]),
        name=f'{variable}',
        showlegend=False,
    ))


#for party in [p for p in PARTIES_LEFT_TO_RIGHT if p in df_melted['Party'].unique()]:
#    fig.add_trace(go.Scatter(
#        x=[None],
#        y=[None],
#        mode='lines+markers',
#        line=dict(color=PARTY2COLOR[party], width=2),
#        marker=dict(symbol='circle', size=8, color=PARTY2COLOR[party]),
#        showlegend=True,
#        name=party,
#    ))


# Update layout for secondary y-axis
fig.update_layout(
    xaxis_title='Number of Favorable Questions Included',
    yaxis_title='Relative Visibility Gain (%)',
    legend_title='Party',
    width=1000,
    height=600,
    legend=dict(
        title=None,
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=-0.1,
        font=dict(size=14),
        #itemwidth=60,
    )
)

In [ ]:
# SAVE VISUALIZATION

save_paper_figure(fig, 'question_favoritism_SG_deluxe_relative_difference_5x3_maxmarkers_circles_custom_legend_times_english', height=360, font_size=18)

## Question Order

In [ ]:
# CALCULATE EMPIRICAL DROP RATES

import statsmodels.api as sm
df_response_frequencies = (~df_voters[(df_voters['questTYPE']==1)].a().isna()).mean().to_frame().rename(columns={0:'all voters'}).reset_index()

x = df_response_frequencies.index
y = df_response_frequencies['all voters']

# Add a constant to the independent variables for intercept calculation
x_with_const = sm.add_constant(x)

model = sm.OLS(y, x_with_const).fit()
slope, intercept = model.params[1], model.params[0]

ic(slope, intercept)

empirical_ols_drop_rates = [1-(intercept + i * slope) for i in range(75)]
empirical_ols_drop_rates

In [ ]:
# VISUALIZE DROP RATES

px.scatter((~df_voters[(df_voters['questTYPE']==1)].a().isna()).mean().to_frame().rename(columns={0:'all voters'}).reset_index(), y='all voters', trendline='ols', trendline_color_override='red', height=800, width=1200)

In [ ]:
def drop_values_randomly(df, drop_rates_dict, seed=None):
    # Set the seed for reproducibility
    if seed is not None:
        np.random.seed(seed)
    
    # Copy the DataFrame to avoid modifying the original
    df_copy = df.copy()
    
    for col, rate in drop_rates_dict.items():
        mask = np.random.rand(len(df_copy)) < rate
        df_copy.loc[mask, col] = np.nan
    return df_copy

In [ ]:
# default Party Visibilitys based on voters that answered all questions
district = 'SG'
df_v_all = df_voters[(df_voters['_district']==district) & (df_voters['N_answers']==75)]
df_c = df_candidates.district(district)

df_v_all = add_candidate_voting_recommendations(df_v_all, df_c)
df_c = df_c.add_recommendation_counts(df_v_all)

df_party_strengths_all_answers = df_c.get_party_visibilities()
df_party_strengths_all_answers

In [ ]:
N_repeats = 10

parties_in_df = [c.split()[0] for c in df_greedy_forward_selection.columns if c.endswith('selection')]
df_question_order_party_strengths = defaultdict(list)

for party in tqdm(parties_in_df):
    ic(party)
    
    for seed in tqdm(range(N_repeats)):
    
        drop_rates_dict = dict(zip(df_voters.a().columns[df_greedy_forward_selection[f'{party} selection'].to_numpy()], empirical_ols_drop_rates))
        df_v_all_dropped = drop_values_randomly(df_v_all, drop_rates_dict, seed=seed)
    
        df_v_all_dropped = add_candidate_voting_recommendations(df_v_all_dropped, df_c)
        df_c = df_c.add_recommendation_counts(df_v_all_dropped)

        modified_party_strength = df_c.get_party_visibilities().loc[party].item()
        df_question_order_party_strengths[party].append(modified_party_strength)

        ic(modified_party_strength)

df_question_order_party_strengths = pd.DataFrame(df_question_order_party_strengths)  
df_question_order_party_strengths

In [ ]:
# RESULTS (RANDOM SAMPLES)
df_question_order_party_strengths

In [ ]:
# BEFORE Party VisibilityS
df_party_strengths_all_answers.iloc[:, 0]

In [ ]:
# AVERAGE DIFFERENCE
(df_question_order_party_strengths - df_party_strengths_all_answers.iloc[:, 0]).mean()

In [ ]:
# STANDARD DEVIATION OF DIFFERENCE
(df_question_order_party_strengths - df_party_strengths_all_answers.iloc[:, 0]).std()

In [ ]:
# AVERAGE RATIO
(df_question_order_party_strengths / df_party_strengths_all_answers.iloc[:, 0]).mean() - 1

In [ ]:
# STANDARD DEVIATION OF RATIO
(df_question_order_party_strengths / df_party_strengths_all_answers.iloc[:, 0]).std()

# Question Type Importance

In [ ]:
df_voters['_party'].unique()

In [ ]:
X = df_voters[df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])].a().to_numpy()

In [ ]:
y = df_voters[df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])]['_party'].to_numpy()

In [ ]:
X = df_voters[(df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])) & (df_voters['questTYPE'] == 1)].a().to_numpy()
y = df_voters[(df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])) & (df_voters['questTYPE'] == 1)]['_party'].to_numpy()

In [ ]:
X = df_voters[(df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])) & (df_voters['N_answers'] == 75)].a().to_numpy()
y = df_voters[(df_voters['_party'].isin([v for v in df_voters['_party'].unique() if v != 'Keine' and v is not None])) & (df_voters['N_answers'] == 75)]['_party'].to_numpy()

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import pandas as pd

rfc = RandomForestClassifier()
rfc.fit(X, y)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgbc = HistGradientBoostingClassifier()
hgbc.fit(X, y)

In [ ]:
df_questions['RFC_feature_importance'] = rfc.feature_importances_
#df_questions['HGBC_feature_importance'] = hgbc.feature_importances_

In [ ]:
df_questions[df_questions['rapide']==1]

In [ ]:
# all
df_questions.groupby('type')['RFC_feature_importance'].describe()

In [ ]:
# questTYPE==1
df_questions.groupby('type')['RFC_feature_importance'].describe()

In [ ]:
# questTYPE==1
df_questions.groupby('type')['RFC_feature_importance'].describe()

In [ ]:
df_questions.groupby('type')[[c for c in df_questions.columns if c.endswith('feature_importance')]].describe()

In [ ]:
model.feature_importances_

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Assuming X_train and y_train are your features and target variable
model = RandomForestClassifier()
model.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [ ]:
df_questions

In [ ]:
qt_boolean_index = np.full_like(question_type_boolean_index, False)
qt_boolean_index[np.random.choice(np.where(question_type_boolean_index)[0], 7)] = True
question_type_boolean_index = qt_boolean_index

In [ ]:
question_type_boolean_index = df_questions['type'].isin(['4-options'])

# randomly sample from question_type
n_samples = 7
qt_boolean_index = np.full_like(question_type_boolean_index, False)
qt_boolean_index[np.random.choice(np.where(question_type_boolean_index)[0], n_samples)] = True
question_type_boolean_index = qt_boolean_index
ic(question_type_boolean_index)

df_voters_question_type = SVDataFrame(df_voters, answer_cols=np.array(get_cols(df_voters))[question_type_boolean_index], weight_cols=np.array(get_cols(df_voters, col_type='weight'))[question_type_boolean_index])
df_candidates_question_type = SVDataFrame(df_candidates, answer_cols=np.array(get_cols(df_candidates))[question_type_boolean_index])

In [ ]:
dfs_party_rank_metrics = []
question_type_boolean_index = df_questions['type'].isin(['4-options'])
n_samples = 7

for _ in tqdm(range(20)):

    # randomly sample from question_type
    qt_boolean_index = np.full_like(question_type_boolean_index, False)
    qt_boolean_index[np.random.choice(np.where(question_type_boolean_index)[0], n_samples)] = True
    question_type_boolean_index = qt_boolean_index
    ic(question_type_boolean_index)
    
    df_voters_question_type = SVDataFrame(df_voters, answer_cols=np.array(get_cols(df_voters))[question_type_boolean_index], weight_cols=np.array(get_cols(df_voters, col_type='weight'))[question_type_boolean_index])
    df_candidates_question_type = SVDataFrame(df_candidates, answer_cols=np.array(get_cols(df_candidates))[question_type_boolean_index])
    
    df_voters_question_type = add_list_voting_recommendations(
        df_voters_question_type, 
        df_candidates_question_type, 
        distance_method='L2_sv', 
        progress_bar=True
    )

    df_voters_question_type.map_lists_to_parties(df_candidates_question_type)
    df_voters_question_type.add_party_ranks(df_candidates_question_type, progress_bar=True)
    dfs_party_rank_metrics.append(df_voters_question_type.get_party_rank_metrics())

In [ ]:
df_voters_recommendations_spc = df_voters_question_type.copy()
for distance_method in tqdm(IMPORTANT_DISTANCE_METHODS):
    print(distance_method)
    df_voters_recommendations_spc = add_candidate_recommendations(
        df_voters_recommendations_spc, 
        df_candidates_question_type, 
        distance_method=distance_method,
        progress_bar=True
    )
    df_to_save = df_voters_recommendations_spc[['recID']+[c for c in df_voters_recommendations_spc.columns if c.startswith('_match')]]
    df_to_save.to_parquet(os.path.join(CACHE_FOLDER, 'df_voters_budget_recommendations_spc_important_methods.parquet'))

In [ ]:
df_voters_list_recommendations = df_voters_question_type.copy()
for distance_method in tqdm(IMPORTANT_DISTANCE_METHODS):
    print(distance_method)
    df_voters_list_recommendations = add_list_recommendations(df_voters_list_recommendations, df_candidates_question_type, distance_method=distance_method, progress_bar=True)
    df_to_save = df_voters_list_recommendations[['recID']+[c for c in df_voters_list_recommendations.columns if c.startswith('_List')]]
    df_to_save.to_parquet(os.path.join(CACHE_FOLDER, 'df_voters_budget_list_recommendations_important_methods.parquet'))

In [ ]:
question_type_boolean_index = df_questions['type'].isin(['7-options'])

df_voters_question_type = SVDataFrame(df_voters, answer_cols=np.array(get_cols(df_voters))[question_type_boolean_index], weight_cols=np.array(get_cols(df_voters, col_type='weight'))[question_type_boolean_index])
df_candidates_question_type = SVDataFrame(df_candidates, answer_cols=np.array(get_cols(df_candidates))[question_type_boolean_index])

In [ ]:
df_voters_recommendations_spc = df_voters_question_type.copy()
for distance_method in tqdm(IMPORTANT_DISTANCE_METHODS):
    print(distance_method)
    df_voters_recommendations_spc = add_candidate_recommendations(
        df_voters_recommendations_spc, 
        df_candidates_question_type, 
        distance_method=distance_method,
        progress_bar=True
    )
    df_to_save = df_voters_recommendations_spc[['recID']+[c for c in df_voters_recommendations_spc.columns if c.startswith('_match')]]
    df_to_save.to_parquet(os.path.join(CACHE_FOLDER, 'df_voters_value_recommendations_spc_important_methods.parquet'))

In [ ]:
df_voters_list_recommendations = df_voters_question_type.copy()
for distance_method in tqdm(IMPORTANT_DISTANCE_METHODS):
    print(distance_method)
    df_voters_list_recommendations = add_list_recommendations(df_voters_list_recommendations, df_candidates_question_type, distance_method=distance_method, progress_bar=True)
    df_to_save = df_voters_list_recommendations[['recID']+[c for c in df_voters_list_recommendations.columns if c.startswith('_List')]]
    df_to_save.to_parquet(os.path.join(CACHE_FOLDER, 'df_voters_value_list_recommendations_important_methods.parquet'))

In [ ]:
question_types = 'policy'
df_voters = df_voters.load_candidate_recommendations(f'df_voters_{question_types}_recommendations_spc_important_methods')
df_voters = df_voters.load_list_recommendations(f'df_voters_{question_types}_list_recommendations_important_methods.parquet')

In [ ]:
df_voters

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# policy
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# policy
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
question_types = 'policy_value'
df_voters = df_voters.load_candidate_recommendations(f'df_voters_{question_types}_recommendations_spc_important_methods')
df_voters = df_voters.load_list_recommendations(f'df_voters_{question_types}_list_recommendations_important_methods.parquet')

In [ ]:
df_voters

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# policy_value
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# policy_value
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
question_types = 'policy_budget'
df_voters = df_voters.load_candidate_recommendations(f'df_voters_{question_types}_recommendations_spc_important_methods')
df_voters = df_voters.load_list_recommendations(f'df_voters_{question_types}_list_recommendations_important_methods.parquet')

In [ ]:
df_voters

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# policy_budget
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# policy_budget
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_voters = df_voters.load_candidate_recommendations(f'df_voters_recommendations_spc_10_methods', selected_distance_methods=IMPORTANT_DISTANCE_METHODS)
df_voters = df_voters.load_list_recommendations(f'df_voters_list_recommendations_10_methods.parquet', selected_distance_methods=IMPORTANT_DISTANCE_METHODS)

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# all
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# all
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
question_types = 'budget'
df_voters = df_voters.load_candidate_recommendations(f'df_voters_{question_types}_recommendations_spc_important_methods')
df_voters = df_voters.load_list_recommendations(f'df_voters_{question_types}_list_recommendations_important_methods.parquet')

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# budget
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# budget
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
question_types = 'value'
df_voters = df_voters.load_candidate_recommendations(f'df_voters_{question_types}_recommendations_spc_important_methods')
df_voters = df_voters.load_list_recommendations(f'df_voters_{question_types}_list_recommendations_important_methods.parquet')

In [ ]:
parties_descending_popularity = list(df_candidates.groupby('_party')['ID_candidate'].count().sort_values(ascending=False).index)

list2party = df_candidates.groupby('ID_list')['_party'].apply(
    lambda x: x[~x.isin(['Parteilos'])].mode()[0] if len(x[~x.isin(['Parteilos'])].mode())==1 
    else sorted(x[~x.isin(['Parteilos'])].mode(), key=parties_descending_popularity.index)[0] if len(x[~x.isin(['Parteilos'])].mode())>1
    else 'Parteilos'
).to_dict()

list2partyrec = df_candidates.groupby('ID_list')['_party_rec'].apply(
    lambda x: x.mode()[0] if len(x.mode())==1 
    else sorted(x.mode(), key=parties_descending_popularity.index)[0]
).to_dict()

In [ ]:
# map lists to parties
for col in [c for c in df_voters.columns if 'ListID' in c]:
    df_voters[col.replace('ListID', 'ListParty')] = df_voters[col].map(list2party.get)
    #df_voters[col.replace('ListID', 'ListPartyRec')] = df_voters[col].map(list2partyrec.get)

df_voters.add_party_ranks(df_candidates, progress_bar=True)

In [ ]:
# value
df_voters.get_party_rank_metrics().style.background_gradient(cmap='Blues_r', axis=0)

In [ ]:
df_candidates = df_candidates.add_recommendation_counts(df_voters)

In [ ]:
# value
df_candidates.get_recommendation_metrics().style.background_gradient(cmap='Blues_r', axis=0)